## 0. Create a ConvNeXt-Tiny model and transforms

In [1]:
import torch
import torchvision
from torch import nn


def create_ConvNeXt_Tiny_Weights_model(num_classes: int = 18, seed: int = 42):
    """Create an ImageNet-pretrained ConvNeXt-Tiny with a new classifier.

    Args:
        num_classes (int): Number of output classes. Defaults to 18.
        seed (int): Seed for the new classifier's initialization. Defaults to 42.

    Returns:
        model (torch.nn.Module): ConvNeXt-Tiny with all parameters trainable.
        transforms: Preprocessing transforms for the pretrained weights.

    Load your trained rock-classification weights and call model.eval()
    before using this model for inference. The new classifier is untrained.
    """
    weights = torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.convnext_tiny(weights=weights)

    # Replace only the final linear layer, preserving normalization and flattening.
    torch.manual_seed(seed)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

    return model, transforms


In [2]:
ConvNeXt_Tiny, ConvNeXt_Tiny_transforms = create_ConvNeXt_Tiny_Weights_model(num_classes=18,seed=42)

In [3]:
from torchinfo import summary

# # Print ConvNeXt_Tiny model summary (uncomment for full output) 

summary(ConvNeXt_Tiny,
        input_size=(1, 3, 224, 224),
         col_names=["input_size", "output_size", "num_params", "trainable"],
         col_width=20,
         row_settings=["var_names"])

Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
ConvNeXt (ConvNeXt)                                     [1, 3, 224, 224]     [1, 18]              --                   True
├─Sequential (features)                                 [1, 3, 224, 224]     [1, 768, 7, 7]       --                   True
│    └─Conv2dNormActivation (0)                         [1, 3, 224, 224]     [1, 96, 56, 56]      --                   True
│    │    └─Conv2d (0)                                  [1, 3, 224, 224]     [1, 96, 56, 56]      4,704                True
│    │    └─LayerNorm2d (1)                             [1, 96, 56, 56]      [1, 96, 56, 56]      192                  True
│    └─Sequential (1)                                   [1, 96, 56, 56]      [1, 96, 56, 56]      --                   True
│    │    └─CNBlock (0)                                 [1, 96, 56, 56]      [1, 96, 56, 56]      79,296               True
│  

## 0.1 create data loaders

In [4]:
from torch.utils.data import DataLoader
from modular.data_setup import create_transformation
from modular.data_setup import RockClassificationDataset

train_dir = "data/carbonate_1223 _merge/PPL-1223/train"
val_dir   = "data/carbonate_1223 _merge/PPL-1223/val"
test_dir  = "data/carbonate_1223 _merge/PPL-1223/test"

from functools import partial
from torchvision import transforms
from torchvision.models import ConvNeXt_Tiny_Weights
from modular.data_setup import remove_bottom_annotation

train_eval_transform = transforms.Compose([
    transforms.Lambda(
        partial(remove_bottom_annotation, pixels=90)
    ),
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1.transforms(),
])

#val_test_transform = train_eval_transform


##########################################
# make train dataset again for inference
##########################################
train_eval_dataset = RockClassificationDataset(
    root_dir=train_dir,
    image_transform=train_eval_transform
)



##########################################
# make test dataset again for inference
##########################################
test_eval_dataset = RockClassificationDataset(
    root_dir=test_dir,
    image_transform=train_eval_transform
)

test_eval_dataloader = DataLoader(
    test_eval_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

# Get the validation transform for testing and Gradio predictions
#val_test_transform = val_dataloader.dataset.image_transform

#absprint(f"train transform is \n{train_transform}")
#print(f"val and test transform is \n{val_test_transform}")
#print(f"mean is \n{mean}")
#print(f"std \n{std}")
#print(f"length of data is  \n{dataset_len}")


# test data
test_dataset = RockClassificationDataset(
    root_dir=test_dir,
    image_transform=train_eval_transform
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

## 1 Gradio

### 1.0 Setting up the model

In [5]:
ConvNeXt_Tiny, ConvNeXt_Tiny_transforms = create_ConvNeXt_Tiny_Weights_model(num_classes=18,seed=42)

ConvNeXt_Tiny.load_state_dict(
    torch.load("models/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth", map_location="cpu", weights_only=True)
)
ConvNeXt_Tiny.eval()

## Put convext_tiny on cpu
ConvNeXt_Tiny.to("cpu") 

# Check the device
next(iter(ConvNeXt_Tiny.parameters())).device

device(type='cpu')

### 1.1 Creating a function to map our inputs and outputs

In [6]:
from typing import Tuple, Dict

def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken.
    """
    # Start the timer
    start_time = timer()
    
    # Transform the target image and add a batch dimension
    img = train_eval_transform(img).unsqueeze(0)
    
    # Put model into evaluation mode and turn on inference mode
    ConvNeXt_Tiny.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(ConvNeXt_Tiny(img), dim=1)
    
    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    
    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)
    
    # Return the prediction dictionary and prediction time 
    return pred_labels_and_probs, pred_time

In [7]:
import random
from PIL import Image
from pathlib import Path
from timeit import default_timer as timer

class_names =train_eval_dataset.classes

# Get a list of all test image filepaths
test_data_paths = list(Path(test_dir).glob("*/*.jpg"))

# Randomly select a test image path
random_image_path = random.sample(test_data_paths, k=1)[0]

# Open the target image
image = Image.open(random_image_path)
print(f"[INFO] Predicting on image at path: {random_image_path}\n")

# Predict on the target image and print out the outputs
pred_dict, pred_time = predict(img=image)
print(f"Prediction label and probability dictionary: \n{pred_dict}")
print(f"Prediction time: {pred_time} seconds")

[INFO] Predicting on image at path: data/carbonate_1223 _merge/PPL-1223/test/class12/Micritic_dolomite.262.jpg

Prediction label and probability dictionary: 
{'class2_class13': 0.003098052693530917, 'class3': 0.0017943830462172627, 'class4': 0.0028886557556688786, 'class5': 0.0005185808404348791, 'class6': 0.0012180841295048594, 'class7': 0.0012429025955498219, 'class8': 0.000760272319894284, 'class9': 0.10786845535039902, 'class10': 0.0005313722067512572, 'class11': 0.005685746669769287, 'class12': 0.34321358799934387, 'class14': 0.00028315995587036014, 'class15': 0.0003448374045547098, 'class16': 0.0005347911501303315, 'class17': 0.0009158809552900493, 'class18_class19': 0.052823927253484726, 'class20': 0.0023659137077629566, 'class21_class22': 0.47391146421432495}
Prediction time: 0.03769 seconds


### 1.2 Create a list of example ofimages

In [8]:
# Create a list of example inputs to our Gradio demo
example_list = [[str(filepath)] for filepath in random.sample(test_data_paths, k=8)]
example_list

[['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class21_class22/class21__Thrombotolite.316.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class18_class19/class19__Stromatolite.420.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class10/Fracture.97.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class21_class22/class21__Thrombotolite.481.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.291.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.245.jpg'],
 ['data/carbonate_1223 _merge/PPL-1223/test/class10/Fracture.74.jpg']]

In [9]:
#[['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class21_class22/class21__Thrombotolite.316.jpg'],
# ['data/carbonate_1223 _merge/PPL-1223/test/class18_class19/class19__Stromatolite.420.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class10/Fracture.97.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class21_class22/class21__Thrombotolite.481.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.291.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.245.jpg'],
 #['data/carbonate_1223 _merge/PPL-1223/test/class10/Fracture.74.jpg']]

more_list=[['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.279_ARS.jpg'], ['data/carbonate_1223 _merge/PPL-1223/test/class2_class13/class2__Bioclastics.328.jpg']]
example_list.extend(more_list)

### 1.3 Building a Gradio interface

In [10]:
import gradio as gr

# Create title, description and article strings
title = "Rock Classification 🪨"
description = "A ConvNeXt_Tiny feature extractor computer vision model to classify images of rocks."
article = "Created at [05_Gradio_on_Huggingface.ipynb](https://github.com/adebam/Rock_Classification)."

# Create the Gradio demo
demo = gr.Interface(fn=predict, # mapping function from input to output
                    inputs=gr.Image(type="pil"), # what are the inputs?
                    outputs=[gr.Label(num_top_classes=5, label="Predictions"), # what are the outputs?
                             gr.Number(label="Prediction time (s)")], # our fn has two outputs, therefore we have two outputs
                    examples=example_list, 
                    title=title,
                    description=description,
                    article=article)

# Launch the demo!
demo.launch(debug=False, # print errors locally?
            share=True) # generate a publically shareable URL?

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a0cf8619f9520f5ddb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 2 Hugging Face

### 2.0 Creating a demos folder to store our Rock Classification app files

In [11]:
import shutil
from pathlib import Path

# Create FoodVision mini demo path
rock_classsification_demo_path = Path("demos/Gradio/rock_classification")

# Remove files that might already exist there and create new directory
if rock_classsification_demo_path.exists():
    shutil.rmtree(rock_classsification_demo_path)
# If the file doesn't exist, create it anyway
rock_classsification_demo_path.mkdir(parents=True, 
                                exist_ok=True)
    
# Check what's in the folder

### 2.1 Creating a folder of example images to use

**Note of the name of the file** <br>
class2__Bioclastics.291: This means it was merged from class2(which is bioclastic) the name of the 2nd class is not there.<br>
Fracture.74.jpg: This is just one class, it was never merged<br>

In [16]:
from pathlib import Path
import shutil
# 1. Create an examples directory
rock_classsification_demo_examples_path = rock_classsification_demo_path / "examples"
rock_classsification_demo_examples_path.mkdir(parents=True, exist_ok=True)

# 2. Collect three random test dataset image paths


# 3. Copy the three random images to the examples directory
for item in example_list:
    src = Path(item[0])
    destination = rock_classsification_demo_examples_path / src.name
    print(f"[INFO] Copying {example} to {destination}")
    shutil.copy2(src=src, dst=destination)

[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Gradio/rock_classification/examples/Finely_crystalline_dolomite.105.jpg
[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Gradio/rock_classification/examples/class21__Thrombotolite.316.jpg
[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Gradio/rock_classification/examples/class19__Stromatolite.420.jpg
[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Gradio/rock_classification/examples/Fracture.97.jpg
[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Gradio/rock_classification/examples/class21__Thrombotolite.481.jpg
[INFO] Copying ['data/carbonate_1223 _merge/PPL-1223/test/class7/Finely_crystalline_dolomite.105.jpg'] to demos/Grad

## 2.3  Moving our trained model to our demo directory

In [19]:
import shutil

# Create a source path for our target model
model_src_path = "models/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth"

# Create a destination path for our target model 
model_destination_path = rock_classsification_demo_path / model_src_path.split("/")[1]

# Try to move the file
try:
    print(f"[INFO] Attempting to move {model_src_path} to {model_destination_path}")
    
    # Move the model
    shutil.copy2(src=model_src_path, 
                dst=model_destination_path)
    
    print(f"[INFO] Model move complete.")

# If the model has already been moved, check if it exists
except:
    print(f"[INFO] No model found at {model_src_path}, perhaps its already been moved, check why?")
    print(f"[INFO] Model exists at {model_destination_path}: {model_destination_path.exists()}")

[INFO] Attempting to move models/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth to demos/Gradio/rock_classification/INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth
[INFO] Model move complete.


### 2.4 Turning our convnext tiny model into a Python script (model.py)

In [22]:
%%writefile demos/Gradio/rock_classification/model.py
import torch
import torchvision

from torch import nn

def create_ConvNeXt_Tiny_Weights_model(num_classes: int = 18, seed: int = 42):
    """Create an ImageNet-pretrained ConvNeXt-Tiny with a new classifier.

    Args:
        num_classes (int): Number of output classes. Defaults to 18.
        seed (int): Seed for the new classifier's initialization. Defaults to 42.

    Returns:
        model (torch.nn.Module): ConvNeXt-Tiny with all parameters trainable.
        transforms: Preprocessing transforms for the pretrained weights.

    Load your trained rock-classification weights and call model.eval()
    before using this model for inference. The new classifier is untrained.
    """
    weights = torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.convnext_tiny(weights=weights)

    # Replace only the final linear layer, preserving normalization and flattening.
    torch.manual_seed(seed)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

    return model, transforms

Writing demos/Gradio/rock_classification/model.py


### 2.5 Turning our rock classification to  a Python script (app.py)

In [28]:
%%writefile demos/Gradio/rock_classification/app.py

#####################################
#        Needs
######################################
from torch.utils.data import DataLoader
from pathlib import Path
from functools import partial
from torchvision import transforms
from torchvision.models import ConvNeXt_Tiny_Weights


APP_DIR = Path(__file__).resolve().parent

def remove_bottom_annotation(image, pixels):
    width, height = image.size

    if not 0 <= pixels < height:
        raise ValueError("Image height must exceed the crop size.")

    return image.crop((0, 0, width, height - pixels))


train_eval_transform = transforms.Compose([
    transforms.Lambda(
        partial(remove_bottom_annotation, pixels=90)
    ),
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1.transforms(),
])

#####################################
#Gradio
######################################
### 1. Imports and class names setup ### 
import gradio as gr
import os
import torch

from model import create_ConvNeXt_Tiny_Weights_model
from timeit import default_timer as timer
from typing import Tuple, Dict

### 2. Model and transforms preparation ###
# convNext model
ConvNeXt_Tiny, ConvNeXt_Tiny_transforms = create_ConvNeXt_Tiny_Weights_model(num_classes=18,seed=42)

ConvNeXt_Tiny.load_state_dict(
    torch.load(APP_DIR/"INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth", map_location="cpu", weights_only=True)
)

### 3. Predict function ###

# Create predict function
def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken.
    """
    # Start the timer
    start_time = timer()
    
    # Transform the target image and add a batch dimension
    img = train_eval_transform(img).unsqueeze(0)
    
    # Put model into evaluation mode and turn on inference mode
    ConvNeXt_Tiny.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(ConvNeXt_Tiny(img), dim=1)
    
    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}
    
    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)
    
    # Return the prediction dictionary and prediction time 
    return pred_labels_and_probs, pred_time
#class list
class_names = [
    "class2_class13", "class3", "class4", "class5",
    "class6", "class7", "class8", "class9",
    "class10", "class11", "class12", "class14",
    "class15", "class16", "class17", "class18_class19",
    "class20", "class21_class22",
]

### 4. Gradio app ###

# Create title, description and article strings
title = "Rock Classification 🪨"
description = "A ConvNeXt_Tiny feature extractor computer vision model to classify images of rocks."
article = "Created at [05_Gradio_on_Huggingface.ipynb](https://github.com/adebam/Rock_Classification)."

# Create examples list from "examples/" directory 
#APP_DIR = Path(__file__).resolve().parent
examples_dir = APP_DIR / "examples"
example_list = [[str(path)] for path in sorted(examples_dir.glob("*.jpg"))]

# Create the Gradio demo
demo = gr.Interface(fn=predict, # mapping function from input to output
                    inputs=gr.Image(type="pil"), # what are the inputs?
                    outputs=[gr.Label(num_top_classes=3, label="Predictions"), # what are the outputs?
                             gr.Number(label="Prediction time (s)")], # our fn has two outputs, therefore we have two outputs
                    # Create examples list from "examples/" directory
                    examples=example_list, 
                    title=title,
                    description=description,
                    article=article)

# Launch the demo!
demo.launch()

Writing demos/Gradio/rock_classification/app.py


### 2.6 Creating a requirements file for FoodVision Mini (requirements.txt)

In [31]:
%%writefile demos/Gradio/rock_classification/requirements.txt
torch==2.14.0
torchvision==0.29.0
gradio==6.27.0
Pillow==12.3.0

Overwriting demos/Gradio/rock_classification/requirements.txt


## 2.7  Downloading our rock classification app files

In [34]:
!ls demos/Gradio/rock_classification/

app.py
examples
INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth
model.py
requirements.txt


In [35]:
# Change into and then zip the foodvision_mini folder but exclude certain files
!cd demos/Gradio/rock_classification/ && zip -r ../rock_classification.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"

# Download the zipped FoodVision Mini app (if running in Google Colab)
try:
    from google.colab import files
    files.download("demos/rock_classification.zip")
except:
    print("Not running in Google Colab, can't use google.colab.files.download(), please manually download.")

  adding: app.py (deflated 59%)
  adding: examples/ (stored 0%)
  adding: examples/Finely_crystalline_dolomite.105.jpg (deflated 1%)
  adding: examples/Fracture.97.jpg (deflated 0%)
  adding: examples/class21__Thrombotolite.316.jpg (deflated 1%)
  adding: examples/class2__Bioclastics.279_ARS.jpg (deflated 0%)
  adding: examples/class2__Bioclastics.245.jpg (deflated 0%)
  adding: examples/class19__Stromatolite.420.jpg (deflated 1%)
  adding: examples/Fracture.74.jpg (deflated 0%)
  adding: examples/class2__Bioclastics.328.jpg (deflated 0%)
  adding: examples/class21__Thrombotolite.481.jpg (deflated 0%)
  adding: examples/class2__Bioclastics.291.jpg (deflated 1%)
  adding: INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth (deflated 7%)
  adding: model.py (deflated 53%)
  adding: requirements.txt (deflated 6%)
Not running in Google Colab, can't use google.colab.files.download(), please manually download.
